# Unit Test Book
### Kumbhar Digvijay Dagadu | IIT Bombay

Master validation notebook for the Volt3X Fault Detection task.

Uses `assert` statements to test **PASS** or **FAIL** on:
- Correctness: right event count, right fault types, right energy value
- Output structure: required columns present
- Physical constraints: energy in realistic range, zero waste for normal samples
- Edge cases: NaN handling, short circuit disambiguation, baseline quality


## 8 Unit Tests — Assert PASS or FAIL

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Running 8 unit tests...\n")
errors = []

try:
    assert 'df' in globals(), "Execution Error: DataFrame 'df' is missing from the global scope."

    # Dynamically find the anomaly mask so we don't get KeyErrors
    if 'is_anomaly' in df.columns:
        is_anom = df['is_anomaly'] == True
    elif 'anomaly' in df.columns:
        is_anom = df['anomaly'] == True
    elif 'fault_type' in df.columns:
        is_anom = df['fault_type'] != 'normal'
    elif 'fault_class' in df.columns:
        is_anom = df['fault_class'] != 'normal'
    else:
        raise ValueError("Could not find an anomaly or fault_type column to evaluate.")

    # Dynamically find the energy variable, OR auto-calculate it from the dataframe!
    energy_val = None
    for var_name in ['total_fault_energy', 'total_wasted_energy_joules', 'total_energy', 'E_total', 'total_E', 'total_waste']:
        if var_name in globals():
            energy_val = globals()[var_name]
            break

    # Auto-Calculate Fallback
    if energy_val is None:
        if 'E_wasted_J' in df.columns:
            energy_val = df.loc[is_anom, 'E_wasted_J'].sum()
        elif 'wasted_power_W' in df.columns:
            energy_val = df.loc[is_anom, 'wasted_power_W'].sum()
        elif 'power_loss_W' in df.columns:
            energy_val = df.loc[is_anom, 'power_loss_W'].sum()
        else:
            raise ValueError("Could not find the total energy variable, and could not find a wasted power column to calculate it from.")

    # T1 — Correct fault event count
    try:
        df_copy = df.copy()
        df_copy['_mask'] = is_anom
        df_copy['_block'] = (df_copy['_mask'] != df_copy['_mask'].shift()).cumsum()
        detected_events = int(df_copy[df_copy['_mask'] == True]['_block'].nunique())
        assert detected_events == 14, f"Expected 14 fault events, got {detected_events}."
        print("T1 PASS  correct_fault_event_count (14 events detected)")
    except Exception as e:
        print(f"T1 FAIL  {e}"); errors.append('T1')

    # T2 — All 5 fault types classified
    try:
        assert 'fault_type' in df.columns or 'fault_class' in df.columns, "Multi-class column missing."
        col_name = 'fault_type' if 'fault_type' in df.columns else 'fault_class'
        required = {'voltage_droop','current_spike','short_circuit','overvoltage','sensor_dropout'}
        detected = set(df[col_name].unique()) - {'normal', 'nominal', 'general_fault'}
        missing  = required - detected
        assert len(missing) == 0, f"Missing fault types: {missing}"
        print("T2 PASS  all_five_fault_types_classified")
    except Exception as e:
        print(f"T2 FAIL  {e}"); errors.append('T2')

    # T3 & T4 — Energy Scope and Boundaries
    try:
        assert energy_val > 0, "Energy is zero or negative."
        assert not np.isnan(energy_val), "Energy is NaN."
        assert 50 <= energy_val <= 500, f"Energy {energy_val:.2f}J is out of physically realistic bounds."
        print(f"T3/T4 PASS  energy_is_valid_and_in_range ({energy_val:.4f} J)")
    except Exception as e:
        print(f"T3/T4 FAIL  {e}"); errors.append('T3/T4')

    # T5 — Normal samples contribute zero waste
    try:
        waste_col = None
        for col in ['wasted_power_W', 'E_wasted_J', 'power_loss_W']:
            if col in df.columns:
                waste_col = col
                break
        if waste_col:
            normal_waste = df.loc[~is_anom, waste_col].sum()
            assert abs(float(normal_waste)) < 1e-6, f"Normal samples contribute {float(normal_waste):.6f} (should be 0)."
            print("T5 PASS  normal_samples_zero_waste")
        else:
            print("T5 SKIP  (No wasted power column found to test normal samples)")
    except Exception as e:
        print(f"T5 FAIL  {e}"); errors.append('T5')

    # T6 — Short circuit identified separately
    try:
        col_name = 'fault_type' if 'fault_type' in df.columns else 'fault_class'
        assert 'short_circuit' in df[col_name].values, "short_circuit not classified."
        print("T6 PASS  short_circuit_identified_separately")
    except Exception as e:
        print(f"T6 FAIL  {e}"); errors.append('T6')

    # T7 — Sensor dropout isolation
    try:
        nan_rows = df[df['voltage_V'].isna() | df['current_A'].isna()]
        if len(nan_rows) > 0:
            assert is_anom[nan_rows.index].all() == True, "NaN rows classified as normal."
        print("T7 PASS  sensor_dropout_not_classified_as_normal")
    except Exception as e:
        print(f"T7 FAIL  {e}"); errors.append('T7')

    # T8 — Baseline not contaminated
    try:
        if 'V_nominal' in globals():
            baseline_v = globals()['V_nominal']
        elif 'baseline_voltage' in df.columns:
            baseline_v = df['baseline_voltage'].mean()
        else:
            baseline_v = df['voltage_V'].mean()

        assert abs(baseline_v - 9.0) <= 0.5, f"Baseline {baseline_v:.4f}V is more than 0.5V from 9.0V — contaminated."
        print(f"T8 PASS  baseline_not_contaminated ({baseline_v:.4f}V)")
    except Exception as e:
        print(f"T8 FAIL  {e}"); errors.append('T8')

except Exception as fatal_e:
    print(f"CRITICAL EXECUTION ERROR: {fatal_e}")
    errors = ['CRITICAL_FAILURE']

print(f"\n{'='*55}")
if 'CRITICAL_FAILURE' not in errors:
    print(f"  Tests passed : {8 - len(errors)}/8")
    if not errors: print("  ALL TESTS PASSED ✓")
print(f"{'='*55}")


Running 8 unit tests...

CRITICAL EXECUTION ERROR: Execution Error: DataFrame 'df' is missing from the global scope.

